#**Research Project**
##**Detecting ASD in Children from Images**

###Conducted by Christel AL HAGE
###With the supervision of Dr. Mounim EL YACOUBI and Dr. Jihyun MUN

##Girls vs Dataset Classification

Dataset used:
https://zenodo.org/records/14890016





In [ ]:
# 1. Install dependencies
!pip install ultralytics transformers tqdm pandas opencv-python

# 2. Download the YOLOv8 Face weights from a stable Hugging Face mirror
!wget -nc -O yolov8n-face.pt https://huggingface.co/junjiang/GestureFace/resolve/main/yolov8n-face.pt

--2026-04-13 08:09:37--  https://huggingface.co/junjiang/GestureFace/resolve/main/yolov8n-face.pt
Resolving huggingface.co (huggingface.co)... 18.244.202.118, 18.244.202.73, 18.244.202.60, ...
Connecting to huggingface.co (huggingface.co)|18.244.202.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67637a2ae1369bd25c05c78b/d9a466380f05487b6bcd11690b1d8d724ede5b7c73aa8648f75c8a78f27229b2?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolov8n-face.pt%3B+filename%3D%22yolov8n-face.pt%22%3B&Expires=1776071377&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzc2MDcxMzc3fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjc2MzdhMmFlMTM2OWJkMjVjMDVjNzhiL2Q5YTQ2NjM4MGYwNTQ4N2I2YmNkMTE2OTBiMWQ4ZDcyNGVkZTViN2M3M2FhODY0OGY3NWM4YTc4ZjI3MjI5YjJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=V92raJ4jedNuddVrztxJUGrOcHa-aZFqavT3Yo2A8ruG-j0%7EbgTwqM

In [ ]:
# The -q means "quiet" so it doesn't spam your output with thousands of lines
!unzip -q ASD_Data.zip -d "ASD Data"

In [ ]:
import os
import torch
import numpy as np
import cv2
import pandas as pd
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from ultralytics import YOLO
from transformers import AutoModelForImageClassification
from tqdm import tqdm

# ==========================================
# 1. HARDWARE SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

# ==========================================
# 2. PREPROCESSING & DATASET SETUP
# ==========================================
# Initialize YOLOv8 Face Model
print("Loading YOLOv8 Face Extractor...")
yolo_face_model = YOLO('yolov8n-face.pt')
yolo_face_model.to(device)

# Custom Laplacian Edge Enhancement Filter
class LaplacianFilter(object):
    def __call__(self, img):
        img_np = np.array(img)
        laplacian = cv2.Laplacian(img_np, cv2.CV_64F)
        sharpened = cv2.addWeighted(img_np.astype(float), 1.5, laplacian, -0.5, 0)
        sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
        return Image.fromarray(sharpened)

# Transform Pipeline (Note: YOLO crops vary in size, so we add a Resize step here)
advanced_transform = transforms.Compose([
    transforms.Resize((224, 224)), # Force YOLO crops to be exactly 224x224
    LaplacianFilter(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset Class using YOLO
class ASDDatasetWithYOLO(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)

    def __getitem__(self, index):
        path, target = self.samples[index]
        # Open image and ensure it is in RGB format
        img = Image.open(path).convert('RGB')

        # Run YOLO inference (verbose=False keeps the console clean)
        results = yolo_face_model(img, verbose=False)
        boxes = results[0].boxes

        # If YOLO finds at least one face
        if len(boxes) > 0:
            # Grab the bounding box with the highest confidence score
            best_box = boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, best_box)

            # Crop the PIL image using the YOLO coordinates
            face_img = img.crop((x1, y1, x2, y2))
        else:
            # Fallback if YOLO somehow misses the face
            face_img = transforms.CenterCrop(224)(img)

        # Apply the Laplacian filter, resizing, and normalization
        if self.transform is not None:
            face_img = self.transform(face_img)

        return face_img, target

# ==========================================
# 3. LOAD THE DATA
# ==========================================
data_dir = './ASD Data'
batch_size = 32

# Instantiate Datasets
train_dataset = ASDDatasetWithYOLO(os.path.join(data_dir, 'Train'), transform=advanced_transform)
val_dataset = ASDDatasetWithYOLO(os.path.join(data_dir, 'valid'), transform=advanced_transform)
test_dataset = ASDDatasetWithYOLO(os.path.join(data_dir, 'Test'), transform=advanced_transform)

# Instantiate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✅ Data Loaded! Discovered Classes: {train_dataset.class_to_idx}")

# ==========================================
# 4. HUGGING FACE DEMOGRAPHIC ANALYSIS
# ==========================================
print("\nLoading Hugging Face Gender Classification Model...")
model_name = "rizvandwiki/gender-classification"
hf_model = AutoModelForImageClassification.from_pretrained(model_name).to(device)
hf_model.eval()

id2label = hf_model.config.id2label
dataset_classes = train_dataset.classes
loaders = {'Train': train_loader, 'Valid': val_loader, 'Test': test_loader}
results = []

with torch.no_grad():
    for split_name, loader in loaders.items():
        print(f"\nAnalyzing {split_name} Set...")

        counts = {cls_name: {label: 0 for label in id2label.values()} for cls_name in dataset_classes}

        for images, targets in tqdm(loader, desc=f"{split_name} Progress"):
            images = images.to(device)
            outputs = hf_model(pixel_values=images)
            preds = torch.argmax(outputs.logits, dim=-1)

            for pred, target in zip(preds, targets):
                gender_label = id2label[pred.item()]
                asd_class = dataset_classes[target.item()]
                counts[asd_class][gender_label] += 1

        for asd_class, gender_counts in counts.items():
            row = {'Dataset Split': split_name, 'Class': asd_class}
            row.update(gender_counts)
            results.append(row)

# ==========================================
# 5. FINAL OUTPUT
# ==========================================
df = pd.DataFrame(results)
print("\n\n=== 📊 Demographic Breakdown Complete ===")
print(df.to_string(index=False))

🚀 Using device: cuda
Loading YOLOv8 Face Extractor...
✅ Data Loaded! Discovered Classes: {'autism': 0, 'tipical': 1}

Loading Hugging Face Gender Classification Model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


Analyzing Train Set...


Train Progress: 100%|██████████| 80/80 [01:09<00:00,  1.15it/s]



Analyzing Valid Set...


Valid Progress: 100%|██████████| 4/4 [00:02<00:00,  1.57it/s]



Analyzing Test Set...


Test Progress: 100%|██████████| 10/10 [00:07<00:00,  1.25it/s]



=== 📊 Demographic Breakdown Complete ===
Dataset Split   Class  female  male
        Train  autism     456   812
        Train tipical     743   525
        Valid  autism      19    31
        Valid tipical      36    14
         Test  autism      56    94
         Test tipical      96    54
